In [ ]:
import umap
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from itertools import combinations
from collections import defaultdict
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler


def merged_lesions_csv(dir_path: str | Path) -> dict[str, pd.DataFrame]:
    """
    Scans a directory for spine lesion radiomics CSV files, groups them by their
    imaging type, and merges them into centralized DataFrames per image modality.
    """
    base_dir = Path(dir_path)

    # 1. Use pathlib's rglob for deep scanning of the target CSV pattern
    all_csv_files = base_dir.rglob("*spine_lesions*.csv")

    # Registry to group file paths by their filename: {csv_name: [Path, Path, ...]}
    csv_groups = defaultdict(list)
    result_dict = {}

    # 2. Group discovered CSV files by their exact filename
    for file_path in all_csv_files:
        csv_groups[file_path.name].append(file_path)

    # 3. Process each group and merge individual patient tables
    for csv_name, file_paths in csv_groups.items():
        loaded_dfs = []

        for file_path in file_paths:
            # Read the target CSV file
            df = pd.read_csv(file_path)

            # Extract the patient folder name (parent directory of the file)
            patient_name = file_path.parent.name

            # Insert the patient identifier safely as the very first column
            df.insert(0, "patient", patient_name)
            loaded_dfs.append(df)

        if not loaded_dfs:
            continue

        # Clean the key name by removing the standard suffix to keep it short
        clean_key_name = csv_name.replace("_radiomics_spine_lesions_features.csv", "")

        # Concatenate all patient rows into one large master DataFrame for this modality
        master_df = pd.concat(loaded_dfs, ignore_index=True)
        result_dict[clean_key_name] = master_df

    return result_dict

In [ ]:
# 1. Global configurations and data loading
pd.set_option('future.no_silent_downcasting', True)
sns.set_theme(style="ticks")

# Load image datasets and clinical mapping files
merged_datasets = merged_lesions_csv(r"E:\DATA_Myelomy")
clinical_df = pd.read_csv(r"E:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")

# Create a clean categorical text Stage mapping column
clinical_df['Stage'] = clinical_df['ISS classification']

# ==================================================================
# 2. DIMENSIONALITY REDUCTION & PLOT PROCESSING PIPELINE
# ==================================================================
for dataset_name, df_raw in merged_datasets.items():
    print(f"Processing PCA for dataset: '{dataset_name}'...")

    # Drop patient metadata to isolate numeric feature matrices
    df_merged = df_raw.merge(
        clinical_df[['Patient ID', 'Stage']],
        left_on='patient',
        right_on='Patient ID',
        how='left'
    ).dropna()

    if df_merged.empty:
        print(f"Skipping {dataset_name}: No valid records remaining after dropping NaNs.")
        continue

    # Isolate feature columns and labels
    df_features = df_merged.drop(columns=['patient', 'Stage', 'Patient ID'])
    stage_labels = df_merged['Stage']

    # Ensure there are columns and rows left to perform PCA
    if df_features.shape[1] < 2 or df_features.shape[0] < 2:
        print(f"Skipping {dataset_name}: Insufficient dimensions for PCA mapping.")
        continue

    # Scale numeric features to unit variance (mean=0, std=1)
    scaled_features = StandardScaler().fit_transform(df_features)

    # Apply 2-Component PCA
    pca = PCA(n_components=2)
    pca_coordinates = pca.fit_transform(scaled_features)

    # Fetch variance ratio values for the axes labels
    pc1_variance = pca.explained_variance_ratio_[0] * 100
    pc2_variance = pca.explained_variance_ratio_[1] * 100

    # Initialize the plot figures
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # --------------------------------------------------------------
    # SUBPLOT 1: Unlabeled Feature Space Structure
    # --------------------------------------------------------------
    axes[0].scatter(
        pca_coordinates[:, 0],
        pca_coordinates[:, 1],
        color="#2b5c8f",
        alpha=0.6,
        edgecolors='w',
        s=50
    )
    axes[0].set_xlabel(f"PC1 ({pc1_variance:.1f} %)", fontsize=11)
    axes[0].set_ylabel(f"PC2 ({pc2_variance:.1f} %)", fontsize=11)
    axes[0].set_title(f"Unlabeled Clusters\nModality: {dataset_name}", fontsize=12, pad=12)

    # --------------------------------------------------------------
    # SUBPLOT 2: Labeled Structural Space Groups (ISS Stages)
    # --------------------------------------------------------------
    # Sort categories alphabetically so Stage 1 appears first in the legend
    sorted_stages = sorted(stage_labels.dropna().unique())

    # Generate an elegant color palette dynamically based on the number of unique stages
    color_palette = sns.color_palette("muted", len(sorted_stages))

    for idx, stage_name in enumerate(sorted_stages):
        # Create a boolean index mask for the target cohort
        mask = stage_labels == stage_name

        axes[1].scatter(
            pca_coordinates[mask, 0],
            pca_coordinates[mask, 1],
            label=str(stage_name),
            color=color_palette[idx],
            alpha=0.8,
            edgecolors='w',
            s=55
        )

    axes[1].set_xlabel(f"PC1 ({pc1_variance:.1f} %)", fontsize=11)
    axes[1].set_ylabel(f"PC2 ({pc2_variance:.1f} %)", fontsize=11)
    axes[1].set_title(f"Labelled Staging Cohorts\nModality: {dataset_name}", fontsize=12, pad=12)
    axes[1].legend(title="Clinical Stage", frameon=True)

    # Apply general cleanup constraints
    sns.despine(fig=fig)
    plt.tight_layout()
    plt.show()
    print("-" * 70)

In [ ]:
# 1. Global style and data loading configurations
pd.set_option('future.no_silent_downcasting', True)
sns.set_theme(style="ticks")

merged_datasets = merged_lesions_csv(r"E:\DATA_Myelomy")
clinical_df = pd.read_csv(r"E:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")

# Capture original categorical text stage groupings
clinical_df['Stage'] = clinical_df['ISS classification']

# ==============================================================================
# 2. t-SNE MANIFOLD LEARNING & PIPELINE PROCESSOR
# ==============================================================================
for dataset_name, df_raw in merged_datasets.items():
    print(f"Processing t-SNE embedding for dataset: '{dataset_name}'...")

    # Cross-reference metrics with staging data arrays
    df_merged = df_raw.merge(
        clinical_df[['Patient ID', 'Stage']],
        left_on='patient',
        right_on='Patient ID',
        how='left'
    ).dropna()

    if df_merged.empty:
        print(f"Skipping {dataset_name}: No valid records matching after dropping NaNs.")
        continue

    # Extract numerical arrays out of tracking metadata frames
    df_features = df_merged.drop(columns=['patient', 'Stage', 'Patient ID'])
    stage_labels = df_merged['Stage']

    # Safeguard dimension boundaries to perform dimensionality projection
    if df_features.shape[1] < 2 or df_features.shape[0] < 2:
        print(f"Skipping {dataset_name}: Insufficient structural sample sizing.")
        continue

    # Standardize predictive arrays to standard unit variance scaling profile
    scaled_features = StandardScaler().fit_transform(df_features)

    # Initialize non-linear t-SNE manifold parameters
    # Adjusting perplexity dynamically if sample size is a tiny
    current_perplexity = min(15, max(5, len(df_merged) - 1))
    tsne = TSNE(perplexity=current_perplexity, learning_rate=200, random_state=42)
    tsne_coordinates = tsne.fit_transform(scaled_features)

    # Initialize figures layout subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # --------------------------------------------------------------------------
    # SUBPLOT 1: Unlabeled Feature Space Structural Topology
    # --------------------------------------------------------------------------
    axes[0].scatter(
        tsne_coordinates[:, 0],
        tsne_coordinates[:, 1],
        color="#2b5c8f",
        alpha=0.6,
        edgecolors='w',
        s=50
    )
    axes[0].set_xlabel("t-SNE Dimension 1", fontsize=11)
    axes[0].set_ylabel("t-SNE Dimension 2", fontsize=11)
    axes[0].set_title(f"Unlabeled Clusters\nModality: {dataset_name}", fontsize=12, pad=12)

    # --------------------------------------------------------------------------
    # SUBPLOT 2: Labeled Structural Space Groups (ISS Stages)
    # --------------------------------------------------------------------------
    # Sort categories chronologically for legend organization
    sorted_stages = sorted(stage_labels.dropna().unique())
    color_palette = sns.color_palette("muted", len(sorted_stages))

    for idx, stage_name in enumerate(sorted_stages):
        # Apply condition indexing row masks to map targeted features cleanly
        mask = stage_labels == stage_name

        axes[1].scatter(
            tsne_coordinates[mask, 0],
            tsne_coordinates[mask, 1],
            label=str(stage_name),
            color=color_palette[idx],
            alpha=0.8,
            edgecolors='w',
            s=55
        )

    axes[1].set_xlabel("t-SNE Dimension 1", fontsize=11)
    axes[1].set_ylabel("t-SNE Dimension 2", fontsize=11)
    axes[1].set_title(f"Labelled Staging Cohorts\nModality: {dataset_name}", fontsize=12, pad=12)
    axes[1].legend(title="Clinical Stage", frameon=True)

    # Apply general visual style layouts cleanup
    sns.despine(fig=fig)
    plt.tight_layout()
    plt.show()
    print("-" * 75)

In [ ]:
# ==============================================================================
# WARNING: To run this script without compatibility crashes, ensure that your
# numpy version is lower than 2.0.0 (e.g., pip install "numpy<2.0.0").
# Newer versions of NumPy alter C-API structures which cause older builds of
# umap-learn / numba to fail with an initialization error.
# ==============================================================================

# 1. Global configurations and data loading
pd.set_option('future.no_silent_downcasting', True)
sns.set_theme(style="ticks")

# Load image datasets and clinical mapping files
merged_datasets = merged_lesions_csv(r"E:\DATA_Myelomy")
clinical_df = pd.read_csv(r"E:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")

# Create a clean categorical text Stage mapping column (No numeric transformation)
clinical_df['Stage'] = clinical_df['ISS classification']

# ==============================================================================
# 2. DIMENSIONALITY REDUCTION & MANIFOLD PLOT PIPELINE
# ==============================================================================
for dataset_name, df_raw in merged_datasets.items():
    print(f"Processing UMAP manifold for dataset: '{dataset_name}'...")

    # Merge radiomics features with clinical patient identifiers
    df_merged = df_raw.merge(
        clinical_df[['Patient ID', 'Stage']],
        left_on='patient',
        right_on='Patient ID',
        how='left'
    ).dropna()

    if df_merged.empty:
        print(f"Skipping {dataset_name}: No valid records remaining after dropping NaNs.")
        continue

    # Isolate metadata columns and isolate strictly numerical features
    df_features = df_merged.drop(columns=['patient', 'Stage', 'Patient ID'])
    df_features = df_features.select_dtypes(include=[np.number])
    stage_labels = df_merged['Stage']

    # Defensive check: ensure there are enough feature spaces to generate an embedding
    if df_features.shape[1] < 2 or df_features.shape[0] < 2:
        print(f"Skipping {dataset_name}: Insufficient dimensions for structural manifold projection.")
        continue

    # Scale numeric features to unit variance (mean=0, std=1)
    scaled_features = StandardScaler().fit_transform(df_features)

    # Initialize and fit UMAP manifold projection parameters
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
    umap_coordinates = reducer.fit_transform(scaled_features)

    # Initialize the modern plot figure subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # --------------------------------------------------------------------------
    # SUBPLOT 1: Unlabeled Feature Space Structure
    # --------------------------------------------------------------------------
    axes[0].scatter(
        umap_coordinates[:, 0],
        umap_coordinates[:, 1],
        color="#1f4e79",
        alpha=0.6,
        edgecolors='w',
        s=50
    )
    axes[0].set_xlabel("UMAP Dimension 1", fontsize=11)
    axes[0].set_ylabel("UMAP Dimension 2", fontsize=11)
    axes[0].set_title(f"Unlabeled Topology\nModality: {dataset_name}", fontsize=12, pad=12)

    # --------------------------------------------------------------------------
    # SUBPLOT 2: Labeled Structural Space Groups (ISS Stages)
    # --------------------------------------------------------------------------
    # Sort categories alphabetically so Stage 1 consistently appears first in legends
    sorted_stages = sorted(stage_labels.dropna().unique())
    color_palette = sns.color_palette("muted", len(sorted_stages))

    for idx, stage_name in enumerate(sorted_stages):
        # Generate mask targeting specific clinical cohort rows
        mask = stage_labels == stage_name

        axes[1].scatter(
            umap_coordinates[mask, 0],
            umap_coordinates[mask, 1],
            label=str(stage_name),
            color=color_palette[idx],
            alpha=0.8,
            edgecolors='w',
            s=55
        )

    axes[1].set_xlabel("UMAP Dimension 1", fontsize=11)
    axes[1].set_ylabel("UMAP Dimension 2", fontsize=11)
    axes[1].set_title(f"Labelled Staging Cohorts\nModality: {dataset_name}", fontsize=12, pad=12)
    axes[1].legend(title="Clinical Stage", frameon=True)

    # Clean styling configurations
    sns.despine(fig=fig)
    plt.tight_layout()
    plt.show()
    print("-" * 75)

In [ ]:
# 1. Global style and data loading configurations
pd.set_option('future.no_silent_downcasting', True)
sns.set_theme(style="ticks")

clinical_df = pd.read_csv(r"E:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")

# Create a clean categorical text target column directly from raw data
clinical_df['Stage'] = clinical_df['ISS classification']

# ==============================================================================
# 2. FEATURE COMBINATION & SELECTIVE ANCHOR VISUALIZATION
# ==============================================================================
for df_name, df_features in merged_lesions_csv(r"E:\DATA_Myelomy").items():
    print(f"Analyzing feature pairing configurations for dataset: '{df_name}'...")

    # Merge clinical target markers with numerical radiomic profiles
    df_combined = pd.merge(
        clinical_df[['Patient ID', 'Stage']],
        df_features,
        left_on='Patient ID',
        right_on='patient',
        how='left'
    )

    # Isolate valid rows containing the categorical target labels
    df_final = df_combined.dropna(subset=['Stage']).reset_index(drop=True)

    if df_final.empty:
        print(f"Skipping {df_name}: No overlapping patient records match.")
        continue

    # Isolate targets (y) and exclusively numerical predictive descriptors (X)
    y_target = df_final['Stage']
    X_features = df_final.select_dtypes(include=['number']).drop(columns=['Stage'], errors='ignore')

    # Impute missing feature array elements safely with zero
    X_features = X_features.fillna(0)
    feature_names = X_features.columns.tolist()

    # Safety check: ensure at least two predictive features exist to create a 2D scatter plot
    if len(feature_names) < 2:
        print(f"Skipping {df_name}: Insufficient number of numeric descriptors.")
        continue

    # Compute ANOVA F-values across all standalone columns relative to class groupings
    f_values, _ = f_classif(X_features, y_target)
    feature_f_score_map = dict(zip(feature_names, f_values))

    best_combined_score = -1
    best_performing_pair = None

    # Evaluate combinations of pairs to identify the dual set with maximum total separation variance
    for feat1, feat2 in combinations(feature_names, 2):
        combined_score = feature_f_score_map[feat1] + feature_f_score_map[feat2]
        if combined_score > best_combined_score:
            best_combined_score = combined_score
            best_performing_pair = (feat1, feat2)

    # --------------------------------------------------------------------------
    # 3. GRAPHICAL DISPLAY GENERATION
    # --------------------------------------------------------------------------
    if best_performing_pair:
        plt.figure(figsize=(9, 6.5))

        # Sort categorical stage classes alphabetically for clean legend distribution
        class_order = sorted(y_target.dropna().unique())

        # Construct spatial cluster representation
        sns.scatterplot(
            data=df_final,
            x=best_performing_pair[0],
            y=best_performing_pair[1],
            hue='Stage',
            hue_order=class_order,
            palette="muted",
            alpha=0.75,
            edgecolors='w',
            s=65
        )

        # Labels, title metadata and axes definitions in English
        plt.xlabel(best_performing_pair[0], fontsize=11)
        plt.ylabel(best_performing_pair[1], fontsize=11)
        plt.title(
            f"Dataset Modality Profile: {df_name}\n"
            f"Top Separating Pair: {best_performing_pair[0]} & {best_performing_pair[1]}\n"
            f"Combined Pairwise F-Score: {best_combined_score:.2f}",
            fontsize=12,
            pad=15
        )

        plt.legend(title="Clinical Stage Group", frameon=True, loc="best")
        sns.despine()
        plt.tight_layout()
        plt.show()
        print(f"Successfully generated plot mapping: {best_performing_pair}\n" + "-" * 75)